# NB08: TEMPORAL ANALYSIS (v16)

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.

# Temporal data availability scenarios (not city tiers!):
# - PREPOST: pre/post composites only (minimum viable)
# - PREPOST_COH: + cross-battle coherence pairs
# - FULL_TEMPORAL: + rolling window stats + z-scores (full stack)
# - POST_ONLY: no pre-event reference (hardest operational scenario)

# CONFIG + GLOBAL SETUP

In [1]:
# @title CELL 3: NB08 CONFIG + GLOBAL SETUP
TIER_SELECTION = [0,1,2]
CITY_SELECTION = None
REQUIRE_UNOSAT = True
FORCE_RERUN = False

import platform, os
if platform.system() == "Windows":
    _setup = r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py"
elif os.path.exists("/content/drive_f"):
    _setup = "/content/drive_f/masterthesis/notebooks/global_setup.py"
else:
    _setup = "/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py"
with open(_setup) as f:
    exec(f.read())

OUT_DIR = RESULTS_ROOT / 'nb08'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"  OUT_DIR: {OUT_DIR}")


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


BDA GLOBAL SETUP
Started: 2026-04-17 22:24:49
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------
  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: True
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:542: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     912.4/7452.0 GB (6539.6 GB free)
  GDrive (F:)     1622.9/3726.0 GB (2103.1 GB free)
  Local data      11507.8/14901.9 GB (3394.1 GB free)
  Data stack      1622.9/3726.0 GB (2103.1 GB free)
  WSL ext4        93.8/1006.9 GB (861.8 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

# CELL S0: PARQUET LOADER + FEATURE CONSTRUCTION

In [2]:
# @title CELL S0: PARQUET LOADER + FEATURE CONSTRUCTION
# =============================================================================
# Loads product_prepost + buildings. Builds feature lists.
# Scene/rolling/block parquets loaded on-demand per experiment cell.
# =============================================================================
import sys, importlib, re, json, time, gc
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.impute import SimpleImputer

print("=" * 70)
print("CELL S0: NB08 PARQUET LOADER")
print("=" * 70)

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
import stack_loader
importlib.reload(stack_loader)
from stack_loader import load_dataset, get_feature_groups

# ---- load product_prepost + buildings (per-tier) ----
_tiers = TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0, 1, 2]
print(f"  Loading tier parquets: {_tiers}")
df_pp = load_tier_parquets(PARQUET_PREPOST_TIER_FMT, _tiers)
df_bldg = load_tier_parquets(PARQUET_BUILDINGS_TIER_FMT, _tiers)

CITIES_TO_PROCESS, _battle_dates = resolve_cities(
    tier_selection=TIER_SELECTION,
    city_selection=[CITY_SELECTION] if isinstance(CITY_SELECTION, str) else CITY_SELECTION,
    require_unosat=REQUIRE_UNOSAT,
)

join_cols = ['building_id', 'city']
bldg_extra = [c for c in df_bldg.columns if c not in df_pp.columns]
df = df_pp.merge(df_bldg[join_cols + bldg_extra], on=join_cols, how='left')
del df_pp
df = df[df['city'].isin(CITIES_TO_PROCESS)].copy()
df = df[df['damage_binary'] >= 0].copy()
print(f"  product_prepost: {len(df)} buildings, {df['city'].nunique()} cities")

TARGET_COL = 'damage_binary'
RANDOM_STATE = 42
RF_PARAMS = {'n_estimators': 200, 'min_samples_leaf': 3, 'random_state': RANDOM_STATE,
             'n_jobs': -1, 'class_weight': 'balanced'}
N_FOLDS = 5

# ---- CARD features (direct column match) ----
card_baseline = [c for c in df.columns if re.match(r's1__(vv|vh)__baseline__', c)]
card_assessment = [c for c in df.columns if re.match(r's1__(vv|vh)__assessment__', c)]
card_prepost = card_baseline + card_assessment
card_post_only = card_assessment

# ---- COH features (direct column match, not group-based) ----
coh_baseline = [c for c in df.columns if re.match(r's1__coh__baseline__', c)]
coh_post_baseline = [c for c in df.columns if re.match(r's1__coh__post_baseline__', c)]
coh_prepost = coh_baseline + coh_post_baseline
coh_post_only = coh_post_baseline
coh_cross = [c for c in df.columns if re.match(r's1__coh.*cross', c)]
coh_all = list(dict.fromkeys(coh_prepost + coh_cross))

# ---- MS + spectral indices ----
feature_groups = get_feature_groups(df)
EXCLUDE_GROUPS = {'meta', 'cloud_freq', 'obs_count', 'other'}
ms_all = []
for gname, cols in feature_groups.items():
    if gname not in EXCLUDE_GROUPS and any(gname.startswith(p) for p in ('comp_', 'lu_', 'landuse', 's2_', 'idx_', 'spectral', 'cd_', 'fire', 'vis')):
        ms_all.extend([c for c in cols if c in df.columns])
ms_all = list(dict.fromkeys(ms_all))
ms_post_only = [c for c in ms_all if any(kw in c for kw in ['post_winter', 'postbattle', 'assessment'])]

# ---- delta features ----
delta_all = [c for c in df.columns if '__delta__' in c or c.startswith('delta_')]

# ---- all ML features from product_prepost ----
all_prepost_feat = [c for c in df.columns if c not in {'building_id', 'city', 'tier', 'damage_binary',
    'damage_label', 'damage', 'ems98_grade', 'unosat_id', 'unosat_date', 'unosat_ep',
    'match_method', 'match_distance', 'in_aoi', 'height', 'num_floors', 'roof_height',
    'area_m2', 'centroid_x', 'centroid_y', 'n_pixels', 'battle_start', 'battle_stop',
    'conflict_ongoing', 'has_card', 'has_coh', 'has_ms'}
    and df[c].dtype.kind in ('f', 'i', 'u')]

# ---- summary ----
print(f"\n  Feature lists:")
print(f"    card_prepost:      {len(card_prepost):>4d} (baseline+assessment)")
print(f"    card_post_only:    {len(card_post_only):>4d}")
print(f"    coh_all:           {len(coh_all):>4d} (baseline+post_baseline+cross)")
print(f"    coh_prepost:       {len(coh_prepost):>4d}")
print(f"    coh_post_only:     {len(coh_post_only):>4d}")
print(f"    coh_cross:         {len(coh_cross):>4d}")
print(f"    ms_all:            {len(ms_all):>4d}")
print(f"    ms_post_only:      {len(ms_post_only):>4d}")
print(f"    delta_all:         {len(delta_all):>4d}")
print(f"    all_prepost_feat:  {len(all_prepost_feat):>4d}")
print(f"    df total cols:     {len(df.columns):>4d}")
print(f"\n  cities={len(CITIES_TO_PROCESS)}: {CITIES_TO_PROCESS}")

# ---- RESULT REGISTRY ----
from bda_results import ResultRegistry
registry = ResultRegistry(RESULTS_ROOT, notebook='NB08a')

# ---- SAVE HELPERS ----
import matplotlib.pyplot as plt
from datetime import datetime as _dt

def save_result(data, name, cell_id, fmt='csv'):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    if fmt == 'csv' and isinstance(data, pd.DataFrame):
        path = cell_dir / f"{name}_{ts}.csv"
        data.to_csv(path, index=False)
    elif fmt == 'json':
        path = cell_dir / f"{name}_{ts}.json"
        import json as _j
        with open(path, 'w') as fh:
            _j.dump(data, fh, indent=2, default=str)
    else:
        raise ValueError(f"Unknown fmt={fmt}")
    print(f"  Saved: {path.relative_to(OUT_DIR)} ({path.stat().st_size / 1024:.1f} KB)")
    return path

def save_fig(fig, name, cell_id, dpi=150):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    path = cell_dir / f"{name}_{ts}.png"
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  Plot: {path.relative_to(OUT_DIR)}")
    return path

# ---- run_experiment helper (used by multiple cells) ----
def run_experiment(feat_df, feat_cols, target_col, groups_col, clf, n_folds=5, impute='median'):
    """Run GroupKFold experiment. Returns dict with auc, f1, y_true, y_proba."""
    row_mask = ~feat_df[feat_cols].isna().all(axis=1)
    df_sub = feat_df[row_mask].copy()
    y = df_sub[target_col].values
    groups = df_sub[groups_col].values
    nan_cols = df_sub[feat_cols].isna().all()
    clean_cols = [c for c in feat_cols if not nan_cols[c]]
    if len(clean_cols) < 2 or len(np.unique(y)) < 2:
        return None
    n_cities = len(np.unique(groups))
    n_f = min(n_folds, n_cities)
    if n_f < 2:
        return None
    imp = SimpleImputer(strategy=impute)
    X = imp.fit_transform(df_sub[clean_cols].values)
    gkf = GroupKFold(n_splits=n_f)
    y_proba = np.full(len(y), np.nan)
    for tr, te in gkf.split(X, y, groups):
        c = clone(clf) if hasattr(clf, 'get_params') else clf
        c.fit(X[tr], y[tr])
        y_proba[te] = c.predict_proba(X[te])[:, 1]
    valid = ~np.isnan(y_proba)
    if valid.sum() < 20:
        return None
    auc = roc_auc_score(y[valid], y_proba[valid])
    f1 = f1_score(y[valid], (y_proba[valid] >= 0.5).astype(int))
    return {'auc': auc, 'f1': f1, 'n_features': len(clean_cols), 'n_buildings': int(valid.sum()),
            'n_cities': n_cities, 'y_true': y[valid], 'y_proba': y_proba[valid],
            'groups': groups[valid], 'feature_cols': clean_cols}

def get_temporal_features(tier, sensor, work_df):
    """Build feature list for a tier x sensor combination from work_df columns."""
    feats = []
    if sensor in ("CARD", "SAR_ALL", "ALL"):
        if tier == "POST_ONLY":
            feats += card_post_only
        elif tier == "FULL_TEMPORAL":
            feats += card_prepost + [c for c in work_df.columns if c.startswith('s1__') and 'roll' in c and 'coh' not in c] + delta_all
        else:
            feats += card_prepost
    if sensor in ("COH", "SAR_ALL", "ALL"):
        if tier == "POST_ONLY":
            feats += coh_post_only
        elif tier in ("PREPOST", "PREPOST_COH"):
            feats += coh_prepost
            if tier == "PREPOST_COH":
                feats += coh_cross
        elif tier == "FULL_TEMPORAL":
            feats += coh_all
            feats += [c for c in work_df.columns if re.match(r's1__coh.*roll', c)]
            feats += [c for c in work_df.columns if 'zscore' in c.lower() and 'coh' in c.lower()]
    if sensor in ("MS", "ALL"):
        if tier == "POST_ONLY":
            feats += ms_post_only
        else:
            feats += ms_all
    return list(dict.fromkeys(c for c in feats if c in work_df.columns))

from sklearn.base import clone
print(f"  Helpers: save_result(), save_fig(), run_experiment(), get_temporal_features(), registry")


CELL S0: NB08 PARQUET LOADER
  Loading tier parquets: [0, 1, 2]
  load_tier_parquets: 3 tiers, 907371 rows
  load_tier_parquets: 3 tiers, 907371 rows
  product_prepost: 598595 buildings, 21 cities

  Feature lists:
    card_prepost:        96 (baseline+assessment)
    card_post_only:      48
    coh_all:             36 (baseline+post_baseline+cross)
    coh_prepost:         36
    coh_post_only:       18
    coh_cross:            0
    ms_all:             192
    ms_post_only:        64
    delta_all:           22
    all_prepost_feat:   346
    df total cols:      372

  cities=21: ['Avdiivka', 'Borodyanka', 'Bucha', 'Chernihiv', 'Chornobaivka', 'Dmytrivka', 'Hostomel', 'Irpin', 'Kharkiv', 'Kherson', 'Kramatorsk', 'Lysychansk', 'Makariv', 'Mariupol', 'Moschun', 'Mykolaiv', 'Okhtyrka', 'Rubizhne', 'Sievierodonetsk', 'Trostianets', 'Volnovakha']
  ResultRegistry: /content/drive_f/masterthesis/results/registry (run_id=20260417_222731)
  Helpers: save_result(), save_fig(), run_experiment(

# CELL ABLATION: DATA SCENARIO x SENSOR GROUP ABLATION MATRIX

In [3]:
# @title CELL ABLATION: DATA SCENARIO x SENSOR GROUP ABLATION MATRIX
# =============================================================================
# Per experiment: build feature list, run RF GroupKFold, save, clear memory.
# Rolling/block parquets loaded on-demand, not kept in memory.
# =============================================================================
import warnings
warnings.filterwarnings("ignore")

print("=" * 70)
print("CELL ABLATION: DATA SCENARIO x SENSOR GROUP ABLATION MATRIX")
print("=" * 70)

# ---- load rolling stats (merge into df, then delete source) ----
card_roll_all = []
try:
    _rs = load_tier_parquets(PARQUET_ROLLING_STATS_TIER_FMT, _tiers, window=7)
    _rs = _rs[_rs['city'].isin(CITIES_TO_PROCESS)]
    rs_feat = [c for c in _rs.columns if c.startswith('s1__') and 'roll' in c]
    card_roll_all = rs_feat  # kept for reference only
    merge_cols = ['building_id', 'city'] + rs_feat
    df_merged = df.merge(_rs[merge_cols].drop_duplicates(), on=['building_id', 'city'], how='left')
    print(f"  rolling_stats roll7: {len(rs_feat)} features merged")
    del _rs
    gc.collect()
except FileNotFoundError:
    df_merged = df.copy()
    print(f"  rolling_stats: NOT FOUND")

# ---- COH rolling from rolling_stats parquet (COH features in same parquet) ----
coh_roll = [c for c in df_merged.columns if re.match(r's1__coh.*roll', c)]
coh_zscore = [c for c in df_merged.columns if 'zscore' in c.lower() and 'coh' in c.lower()]

# get_temporal_features defined in S0, takes work_df param

TEMPORAL_TIERS = ["PREPOST", "PREPOST_COH", "FULL_TEMPORAL", "POST_ONLY"]
SENSOR_GROUPS = ["CARD", "COH", "MS", "SAR_ALL", "ALL"]

# feature count matrix
print(f"\n  Feature count matrix:")
print(f"  {'':12s}", end="")
for s in SENSOR_GROUPS:
    print(f"{s:>10s}", end="")
print()
for t in TEMPORAL_TIERS:
    print(f"  {t:10s}", end="")
    for s in SENSOR_GROUPS:
        feats = get_temporal_features(t, s, df_merged)
        print(f"{len(feats):10d}", end="")
    print()

# run experiments
rf = RandomForestClassifier(**RF_PARAMS)
all_results = []

for tier in TEMPORAL_TIERS:
    for sensor in SENSOR_GROUPS:
        key = f"{tier}_{sensor}"
        feat_cols = get_temporal_features(tier, sensor, df_merged)
        if len(feat_cols) < 2:
            print(f"    {key:25s}: SKIP (< 2 features)")
            all_results.append({'tier': tier, 'sensor': sensor, 'auc': np.nan, 'f1': np.nan, 'n_features': 0})
            continue

        result = run_experiment(df_merged, feat_cols, TARGET_COL, 'city', rf, N_FOLDS)
        if result is None:
            print(f"    {key:25s}: SKIP (insufficient data)")
            all_results.append({'tier': tier, 'sensor': sensor, 'auc': np.nan, 'f1': np.nan, 'n_features': len(feat_cols)})
            continue

        print(f"    {key:25s}: AUC={result['auc']:.3f}  F1={result['f1']:.3f}  nfeat={result['n_features']}  n={result['n_buildings']}")
        all_results.append({'tier': tier, 'sensor': sensor, **{k: result[k] for k in ('auc', 'f1', 'n_features', 'n_buildings', 'n_cities')}})

        registry.log_experiment(
            cell_id='cell_ablation', experiment_name=f'ablation_{key}',
            parquet_name='bda_product_prepost', tier_selection=_tiers,
            classifier_name='RF-200', classifier_params=RF_PARAMS,
            feature_set_name=f'{tier}_{sensor}', feature_cols=result['feature_cols'],
            cv_method='GroupKFold', n_folds=N_FOLDS, imputation='median',
            y_true=result['y_true'], y_proba=result['y_proba'], groups=result['groups'],
            note=f'NB08a ablation: tier={tier} sensor={sensor}',
            tags=['ablation', tier, sensor],
        )
        gc.collect()

# cleanup merged df
del df_merged
gc.collect()

# AUC matrix
results_df = pd.DataFrame(all_results)
save_result(results_df, 'ablation_matrix', 'cell_ablation')

print(f"\n  AUC MATRIX:")
print(f"  {'':15s}", end="")
for s in SENSOR_GROUPS:
    print(f"{s:>10s}", end="")
print()
for t in TEMPORAL_TIERS:
    print(f"  {t:13s}", end="")
    for s in SENSOR_GROUPS:
        row = results_df[(results_df['tier'] == t) & (results_df['sensor'] == s)]
        auc = row['auc'].values[0] if len(row) > 0 else np.nan
        if np.isnan(auc):
            print(f"{'---':>10s}", end="")
        else:
            print(f"{auc:>10.3f}", end="")
    print()

print(f"\n{'='*70}")
print("CELL ABLATION COMPLETE")
print(f"{'='*70}")


CELL ABLATION: DATA SCENARIO x SENSOR GROUP ABLATION MATRIX
  load_tier_parquets: 3 tiers, 907371 rows
  rolling_stats roll7: 48 features merged

  Feature count matrix:
                    CARD       COH        MS   SAR_ALL       ALL
  PREPOST           96        36       192       132       324
  PREPOST_COH        96        36       192       132       324
  FULL_TEMPORAL       166        36       192       202       394
  POST_ONLY         48        18        64        66       130
    PREPOST_CARD             : AUC=0.525  F1=0.002  nfeat=96  n=368288
  REG: ablation_PREPOST_CARD                         AUC=0.5248 F1=0.0018 n=368288 feat=96 cities=21 [NB08a/cell_ablation]
    PREPOST_COH              : AUC=0.583  F1=0.062  nfeat=36  n=163644
  REG: ablation_PREPOST_COH                          AUC=0.5826 F1=0.0621 n=163644 feat=36 cities=16 [NB08a/cell_ablation]
    PREPOST_MS               : AUC=0.630  F1=0.001  nfeat=192  n=310131
  REG: ablation_PREPOST_MS                       

# CELL CLASSIFIERS: TOP CLASSIFIERS x DATA SCENARIOS

In [ ]:
# @title CELL CLASSIFIERS: TOP CLASSIFIERS x DATA SCENARIOS
# =============================================================================
# Tests top classifiers across temporal tiers using ALL sensor group.
# Answers: "do advanced classifiers compensate for sparse temporal data?"
# Parquet: same df as T1 (product_prepost + merged rolling_stats).
# =============================================================================
import warnings
warnings.filterwarnings("ignore")
from sklearn.base import clone

print("=" * 70)
print("CELL CLASSIFIERS: TOP CLASSIFIERS x DATA SCENARIOS")
print("=" * 70)

# merge rolling stats for FULL_TEMPORAL tier
try:
    _rs = load_tier_parquets(PARQUET_ROLLING_STATS_TIER_FMT, _tiers, window=7)
    _rs = _rs[_rs['city'].isin(CITIES_TO_PROCESS)]
    rs_feat = [c for c in _rs.columns if c.startswith('s1__') and 'roll' in c]
    df_merged = df.merge(_rs[['building_id', 'city'] + rs_feat].drop_duplicates(), on=['building_id', 'city'], how='left')
    del _rs; gc.collect()
    print(f"  rolling_stats merged: {len(rs_feat)} features")
except FileNotFoundError:
    df_merged = df
    print(f"  rolling_stats: NOT FOUND, using df without rolling")

classifiers = {
    'RF_200': RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE,
                                      n_jobs=-1, class_weight='balanced'),
}

try:
    from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier
    classifiers['ExtraTrees'] = ExtraTreesClassifier(n_estimators=200, min_samples_leaf=3,
                                                      random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')
    classifiers['GBM'] = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
                                                     random_state=RANDOM_STATE)
except: pass

try:
    from lightgbm import LGBMClassifier
    classifiers['LightGBM'] = LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                              class_weight='balanced', verbose=-1)
except: pass

try:
    from xgboost import XGBClassifier
    classifiers['XGBoost'] = XGBClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                            scale_pos_weight=5, eval_metric='logloss', verbosity=0)
except: pass

print(f"  Classifiers: {list(classifiers.keys())}")

TIERS = ["PREPOST", "PREPOST_COH", "FULL_TEMPORAL", "POST_ONLY"]
df_valid = df_merged[df_merged[TARGET_COL] >= 0].copy()
results_t2 = []

for tier in TIERS:
    feat_cols = get_temporal_features(tier, "ALL", df_merged)
    if len(feat_cols) < 2:
        continue

    row_mask = ~df_valid[feat_cols].isna().all(axis=1)
    df_sub = df_valid[row_mask]
    y_sub = df_sub[TARGET_COL].values
    groups_sub = df_sub['city'].values

    nan_col_mask = df_sub[feat_cols].isna().all()
    feat_clean = [c for c in feat_cols if not nan_col_mask[c]]
    if len(feat_clean) < 2:
        continue

    n_folds = min(N_FOLDS, len(np.unique(groups_sub)))
    if n_folds < 2:
        continue

    imp = SimpleImputer(strategy='median')
    X = imp.fit_transform(df_sub[feat_clean].values)
    gkf = GroupKFold(n_splits=n_folds)

    for clf_name, clf in classifiers.items():
        y_proba = np.full(len(y_sub), np.nan)
        try:
            for train_idx, test_idx in gkf.split(X, y_sub, groups_sub):
                clf_copy = clone(clf)
                clf_copy.fit(X[train_idx], y_sub[train_idx])
                if hasattr(clf_copy, 'predict_proba'):
                    y_proba[test_idx] = clf_copy.predict_proba(X[test_idx])[:, 1]
                else:
                    raw = clf_copy.decision_function(X[test_idx])
                    y_proba[test_idx] = 1.0 / (1.0 + np.exp(-raw))

            valid = ~np.isnan(y_proba)
            auc = roc_auc_score(y_sub[valid], y_proba[valid])
            f1 = f1_score(y_sub[valid], (y_proba[valid] >= 0.5).astype(int))
        except Exception as e:
            auc, f1 = float('nan'), float('nan')
            print(f"  WARNING {tier} x {clf_name}: {e}")

        results_t2.append({'tier': tier, 'classifier': clf_name, 'auc': auc, 'f1': f1,
                           'n_features': len(feat_clean)})
        if not np.isnan(auc):
            print(f"  {tier} x {clf_name:15s}: AUC={auc:.3f}  F1={f1:.3f}  nfeat={len(feat_clean)}")

            registry.log_experiment(
                cell_id='cell_classifiers',
                experiment_name=f'clf_{tier}_{clf_name}',
                parquet_name='bda_product_prepost',
                tier_selection=TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0,1,2],
                classifier_name=clf_name,
                feature_set_name=f'{tier}_ALL',
                feature_cols=feat_clean,
                cv_method='GroupKFold',
                n_folds=n_folds,
                imputation='median',
                y_true=y_sub[valid],
                y_proba=y_proba[valid],
                groups=groups_sub[valid],
                note=f'NB08a classifier comparison: tier={tier} clf={clf_name}',
                tags=['classifier_comparison', tier, clf_name],
            )

if results_t2:
    t2_df = pd.DataFrame(results_t2)
    pivot = t2_df.pivot(index='classifier', columns='tier', values='auc')
    print(f"\n  AUC matrix (classifier x tier):")
    print(pivot.to_string(float_format='{:.3f}'.format))

    print(f"\n  Best classifier per tier:")
    for tier in TIERS:
        rows = t2_df[t2_df['tier'] == tier]
        if len(rows) > 0 and rows['auc'].notna().any():
            best = rows.loc[rows['auc'].idxmax()]
            print(f"    {tier}: {best['classifier']:15s} AUC={best['auc']:.3f}")
        elif len(rows) > 0:
            print(f"    {tier}: all NaN (no valid folds)")



# save results
if results_t2:
    save_result(t2_df, 'classifiers_x_tiers', 'cell_classifiers')

    # heatmap
    import matplotlib.pyplot as plt, seaborn as sns
    pivot = t2_df.pivot(index='classifier', columns='tier', values='auc')
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.5, vmax=1.0, ax=ax)
    ax.set_title('AUC: Classifier x Temporal Tier (ALL sensors, GroupKFold)')
    save_fig(fig, 'classifier_x_tier_heatmap', 'cell_classifiers')
else:
    print("  Heatmap skipped: no valid AUC values")

# save registry
registry.save()

del df_merged, df_valid
gc.collect()


CELL CLASSIFIERS: TOP CLASSIFIERS x DATA SCENARIOS
  load_tier_parquets: 3 tiers, 907371 rows
  rolling_stats merged: 48 features
  Classifiers: ['RF_200', 'ExtraTrees', 'GBM', 'LightGBM', 'XGBoost']
  PREPOST x RF_200         : AUC=0.657  F1=0.001  nfeat=324
  REG: clf_PREPOST_RF_200                            AUC=0.6571 F1=0.0009 n=368288 feat=324 cities=21 [NB08a/cell_classifiers]
  PREPOST x ExtraTrees     : AUC=0.718  F1=0.001  nfeat=324
  REG: clf_PREPOST_ExtraTrees                        AUC=0.7178 F1=0.0012 n=368288 feat=324 cities=21 [NB08a/cell_classifiers]


# CELL INVENTORY: CROSS-CITY DATA INVENTORY

In [ ]:
# @title CELL INVENTORY: CROSS-CITY DATA INVENTORY
# =============================================================================
# Inventories what temporal products are available per city.
# Shows which temporal tiers each city can support.
# Parquet: product_prepost (feature availability = non-NaN columns per city).
# =============================================================================
print("=" * 70)
print("CELL INVENTORY: CROSS-CITY DATA INVENTORY")
print("=" * 70)

inventory = []
for city_name in sorted(df['city'].unique()):
    city_df = df[df['city'] == city_name]
    n_bldg = len(city_df)
    n_dmg = (city_df[TARGET_COL] == 1).sum()

    card_n = sum(1 for c in card_prepost if c in city_df.columns and city_df[c].notna().any())
    coh_n = sum(1 for c in coh_prepost if c in city_df.columns and city_df[c].notna().any())
    coh_cx = sum(1 for c in coh_cross if c in city_df.columns and city_df[c].notna().any())
    roll_n = sum(1 for c in card_roll_all if c in city_df.columns and city_df[c].notna().any())
    ms_n = sum(1 for c in ms_all if c in city_df.columns and city_df[c].notna().any())

    if roll_n > 0:
        max_tier = "FULL_TEMPORAL"
    elif coh_cx > 0:
        max_tier = "PREPOST_COH"
    elif card_n > 0 or coh_n > 0 or ms_n > 0:
        max_tier = "PREPOST"
    else:
        max_tier = "POST_ONLY"

    inventory.append({'city': city_name, 'n_bldg': n_bldg, 'n_dmg': n_dmg,
                      'CARD': card_n, 'COH': coh_n, 'COH_x': coh_cx,
                      'ROLL': roll_n, 'MS': ms_n, 'tier': max_tier})

inv_df = pd.DataFrame(inventory)
print(f"\n  {'City':25s} {'Bldg':>7s} {'Dmg':>5s} {'CARD':>5s} {'COH':>5s} {'CrB':>5s} {'Roll':>5s} {'MS':>5s} {'Tier':>5s}")
print(f"  {'-'*80}")
for _, row in inv_df.iterrows():
    print(f"  {row['city']:25s} {row['n_bldg']:7d} {row['n_dmg']:5d} {row['CARD']:5d} {row['COH']:5d} {row['COH_x']:5d} {row['ROLL']:5d} {row['MS']:5d} {row['tier']:>5s}")

print(f"\n  Tier distribution: {inv_df['tier'].value_counts().to_dict()}")



# save results
save_result(inv_df, 'cross_city_inventory', 'cell_inventory')


# CELL PROFILES: TEMPORAL PROFILES FROM SCENE PARQUETS

Per-city observation density, NaN structure at source level. Explains WHY some product_prepost features are NaN.

In [ ]:
# @title CELL PROFILES: TEMPORAL PROFILES FROM SCENE PARQUETS
# =============================================================================
# Reads per-date scene parquets to show observation counts per city per period.
# Flags cities with < 3 observations per period (7-stat aggregation unreliable).
# Parquet: bda_scene_card, bda_scene_coh (per-date long-format).
# =============================================================================
print("=" * 70)
print("CELL PROFILES: TEMPORAL PROFILES FROM SCENE PARQUETS")
print("=" * 70)

# Load scene parquets on-demand (not kept in S0)
_scene_card = None
_scene_coh = None
try:
    _scene_card = load_tier_parquets(PARQUET_SCENE_CARD_TIER_FMT, _tiers)
    _scene_card = _scene_card[_scene_card['city'].isin(CITIES_TO_PROCESS)]
    print(f"  scene_card: {len(_scene_card):,} rows")
except FileNotFoundError:
    print(f"  scene_card: NOT FOUND")
try:
    _scene_coh = load_tier_parquets(PARQUET_SCENE_COH_TIER_FMT, _tiers)
    _scene_coh = _scene_coh[_scene_coh['city'].isin(CITIES_TO_PROCESS)]
    print(f"  scene_coh: {len(_scene_coh):,} rows")
except FileNotFoundError:
    print(f"  scene_coh: NOT FOUND")

for label, sdf in [('CARD', _scene_card), ('COH', _scene_coh)]:
    if sdf is None:
        print(f"\n  {label}: NOT LOADED")
        continue

    date_col = 'date2' if 'date2' in sdf.columns else 'date'
    has_ts = 'timestep' in sdf.columns
    has_pl = 'period_label' in sdf.columns

    print(f"\n  {label}: {len(sdf):,} rows, {sdf['city'].nunique()} cities")
    print(f"  {'City':<22s} {'Dates':>6s} {'Bldgs':>7s}", end="")
    if has_ts:
        print(f" {'t_min':>6s} {'t_max':>6s}", end="")
    if has_pl:
        print(f" {'pre':>5s} {'cross':>5s} {'post':>5s}", end="")
    print()

    nan_risks = []
    for city in sorted(sdf['city'].unique()):
        csdf = sdf[sdf['city'] == city]
        n_dates = csdf[date_col].nunique()
        n_bldg = csdf['building_id'].nunique()
        print(f"  {city:<22s} {n_dates:>6d} {n_bldg:>7d}", end="")
        if has_ts:
            print(f" {int(csdf['timestep'].min()):>6d} {int(csdf['timestep'].max()):>6d}", end="")
        if has_pl:
            for period in ['prebattle', 'crossbattle', 'postbattle']:
                n = csdf[csdf['period_label'] == period][date_col].nunique()
                print(f" {n:>5d}", end="")
                if 0 < n < 3:
                    nan_risks.append(f"{city} {label} {period}: {n} dates")
                elif n == 0:
                    nan_risks.append(f"{city} {label} {period}: ZERO dates")
        print()

    if nan_risks:
        print(f"\n  NaN RISK (< 3 dates -> unreliable 7-stat aggregation):")
        for r in nan_risks:
            print(f"    {r}")



# save results to CSV
t4_rows = []
# Load scene parquets on-demand (not kept in S0)
_scene_card = None
_scene_coh = None
try:
    _scene_card = load_tier_parquets(PARQUET_SCENE_CARD_TIER_FMT, _tiers)
    _scene_card = _scene_card[_scene_card['city'].isin(CITIES_TO_PROCESS)]
    print(f"  scene_card: {len(_scene_card):,} rows")
except FileNotFoundError:
    print(f"  scene_card: NOT FOUND")
try:
    _scene_coh = load_tier_parquets(PARQUET_SCENE_COH_TIER_FMT, _tiers)
    _scene_coh = _scene_coh[_scene_coh['city'].isin(CITIES_TO_PROCESS)]
    print(f"  scene_coh: {len(_scene_coh):,} rows")
except FileNotFoundError:
    print(f"  scene_coh: NOT FOUND")

for label, sdf in [('CARD', _scene_card), ('COH', _scene_coh)]:
    if sdf is None:
        continue
    date_col = 'date2' if 'date2' in sdf.columns else 'date'
    has_pl = 'period_label' in sdf.columns
    for city in sorted(sdf['city'].unique()):
        csdf = sdf[sdf['city'] == city]
        row = {'modality': label, 'city': city,
               'n_dates': csdf[date_col].nunique(),
               'n_buildings': csdf['building_id'].nunique()}
        if has_pl:
            for period in ['prebattle', 'crossbattle', 'postbattle']:
                row[f'n_dates_{period}'] = csdf[csdf['period_label'] == period][date_col].nunique()
        t4_rows.append(row)
if t4_rows:
    save_result(pd.DataFrame(t4_rows), 'temporal_profiles', 'cell_profiles')

del _scene_card, _scene_coh
gc.collect()


# CELL PRODUCTS: PRODUCT COMPARISON

Compares raw P1, rolling_stats, and block_stats features head-to-head per city.

In [ ]:
# @title CELL PRODUCTS: PRODUCT COMPARISON (raw P1 vs rolling vs blocks)
# =============================================================================
# Quick RF (50 trees, 3-fold stratified CV) per product per city.
# Compares: raw_P1, roll2/3/5/7/13 stats, block post-invasion, block baseline.
# Parquets: bda_product_prepost, bda_rolling_stats_card, bda_block_stats_card.
# Uses StratifiedKFold PER CITY (not GroupKFold) since comparison is within-city.
# =============================================================================
import re

print("=" * 70)
print("CELL PRODUCTS: PRODUCT COMPARISON")
print("=" * 70)

label_df = df[['building_id', 'city', TARGET_COL]].drop_duplicates()
products = {}

# raw P1
if card_prepost:
    products['raw_P1'] = {'df': df, 'cols': card_prepost}

# rolling stats per window size — load on demand
for _ws in [3, 7, 13]:
    try:
        _rs = load_tier_parquets(PARQUET_ROLLING_STATS_TIER_FMT, _tiers, window=_ws)
        _rs = _rs[_rs['city'].isin(CITIES_TO_PROCESS)]
        rs_df = _rs.merge(label_df, on=['building_id', 'city'], how='inner')
        ws_cols = [c for c in rs_df.columns if c.startswith('s1__') and f'roll{_ws}' in c]
        if ws_cols:
            products[f'roll{_ws}'] = {'df': rs_df, 'cols': ws_cols}
        del _rs
    except FileNotFoundError:
        pass

# block stats — load on demand
try:
    _bs = load_tier_parquets(PARQUET_BLOCK_STATS_TIER_FMT, _tiers)
    _bs = _bs[_bs['city'].isin(CITIES_TO_PROCESS)]
    bs_df = _bs.merge(label_df, on=['building_id', 'city'], how='inner')
    del _bs
    bs_cols = [c for c in bs_df.columns if c.startswith('s1__') and 'blk' in c]
    post_blk = [c for c in bs_cols if re.search(r'__blk\d+__', c) and '__blk00__' not in c]
    if post_blk:
        products['blk_post'] = {'df': bs_df, 'cols': post_blk}
    bl_cols = [c for c in bs_cols if '__blk00__' in c]
    if bl_cols:
        products['blk_baseline'] = {'df': bs_df, 'cols': bl_cols}

if not products:
    print("  No products available")
else:
    print(f"  Products: {list(products.keys())}")
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    auc_rows = []

    for prod_name, pinfo in products.items():
        pdf = pinfo['df']
        feat = [c for c in pinfo['cols'] if c in pdf.columns]
        if len(feat) < 2:
            continue

        for city in sorted(pdf['city'].unique()):
            cdf = pdf[pdf['city'] == city]
            avail = [c for c in feat if cdf[c].notna().sum() > 20]
            if len(avail) < 2:
                continue
            y = cdf[TARGET_COL].values
            if len(np.unique(y)) < 2 or len(y) < 30:
                continue

            imp = SimpleImputer(strategy='median')
            X = imp.fit_transform(cdf[avail].values)
            rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, class_weight='balanced')
            yp = np.zeros(len(y))
            for tr, te in skf.split(X, y):
                rf.fit(X[tr], y[tr])
                yp[te] = rf.predict_proba(X[te])[:, 1]
            auc = roc_auc_score(y, yp)
            auc_rows.append({'product': prod_name, 'city': city, 'auc': auc, 'n_feat': len(avail)})

    if auc_rows:
        auc_df = pd.DataFrame(auc_rows)
        pivot = auc_df.pivot(index='city', columns='product', values='auc')
        print(f"\n  AUC per city x product:")
        print(pivot.to_string(float_format='{:.3f}'.format))

        print(f"\n  Mean AUC across cities:")
        means = auc_df.groupby('product')['auc'].agg(['mean', 'std']).sort_values('mean', ascending=False)
        for prod, row in means.iterrows():
            print(f"    {prod:<20s}: {row['mean']:.3f} (+/-{row['std']:.3f})")

        if len(pivot.columns) >= 2:
            print(f"\n  Product correlation (Spearman):")
            corr = pivot.corr(method='spearman')
            print(corr.to_string(float_format='{:.2f}'.format))
    else:
        print("  No AUC results")



# save results + plot
if auc_rows:
    save_result(auc_df, 'product_comparison', 'cell_products')

    # bar chart
    import matplotlib.pyplot as plt
    means = auc_df.groupby('product')['auc'].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    means.plot.bar(ax=ax, color='steelblue', edgecolor='black')
    ax.set_ylabel('Mean AUC')
    ax.set_title('Product Comparison: Mean AUC across cities')
    ax.set_ylim(0.5, 1.0)
    ax.axhline(0.813, color='red', linestyle='--', label='Dietrich AUC=0.813')
    ax.legend()
    save_fig(fig, 'product_comparison_bar', 'cell_products')
